# Results — the scorecard

Read the failures honestly. P1 (Holcus) is the only load-bearing prediction. P2–P5 are secondary; a falsified secondary stays in the data.

In [1]:
import sys, os, json, math
sys.path.insert(0, os.path.join(os.getcwd(), 'engine'))
import maths as M
from tools import DeSitterCavitationModule
E = DeSitterCavitationModule()
print('engine', E.version, '—', E.display_name)


engine 0.100 — De Sitter Cavitation Engine — No Singularity: the Abrikosov-Vortex Core


## P1 — HOLCUS: `K_core(M) = (3/2) c⁸/(G⁴M⁴)`

In [2]:
chk = E.run('no_singularity_check', {})['result']
print('finite everywhere .............', chk['finite'])
print('K_core proportional to M^-4 ...', chk['scales_as_M_minus_4'],
      '  (max rel err', max(d['rel_err'] for d in chk['m4_detail']), ')')
print('sub-Planckian above crossover .', chk['sub_planckian_above_crossover'],
      '  crossover =', round(chk['M_crossover_over_mPl'], 4), 'm_Pl')
print('Schwarzschild K -> inf (contrast)', chk['schwarzschild_diverges_at_r0'])
print()
print('P1 VERDICT:', 'STANDS' if chk['PASS'] else 'DEAD')

finite everywhere ............. True
K_core proportional to M^-4 ... True   (max rel err 1.6430465043699702e-16 )
sub-Planckian above crossover . True   crossover = 1.1067 m_Pl
Schwarzschild K -> inf (contrast) True

P1 VERDICT: STANDS


In [3]:
# the mass sweep, both formulas
Msun = M.M_SUN
print(f"{'M [Msun]':>12s}{'K 24/r_s^4':>16s}{'K closed':>16s}{'rel':>10s}{'K/K_Planck':>14s}")
for e in (-19, -10, 0, 1, 4, 9):
    Mk = (10.0**e)*Msun if e != -19 else 1e12
    a = M.kretschmann_core(Mk); b = M.kretschmann_core_closed(Mk)
    tag = '1e12 kg' if e == -19 else f'1e{e}'
    print(f'{tag:>12s}{a:16.4e}{b:16.4e}{abs(a-b)/a:10.1e}{a/M.K_PLANCK:14.3e}')

    M [Msun]      K 24/r_s^4        K closed       rel    K/K_Planck
     1e12 kg      4.9321e+60      4.9321e+60   0.0e+00     3.366e-79
       1e-10      3.1518e+27      3.1518e+27   1.7e-16    2.151e-112
         1e0      3.1518e-13      3.1518e-13   1.6e-16    2.151e-152
         1e1      3.1518e-17      3.1518e-17   5.9e-16    2.151e-156
         1e4      3.1518e-29      3.1518e-29   1.8e-16    2.151e-168
         1e9      3.1518e-49      3.1518e-49   3.6e-16    2.151e-188


## Engineering table (the paper's engineering portion)

In [4]:
rows = E.run('mass_class_table', {})['result']
for row in rows:
    print(f"{row['class']:26s}  r_s={row['r_s_m']:.2e} m  tau={row['tau_interior_s']:.2e} s  "
          f"T_H={row['T_hawking_K']:.2e} K  t_evap={row['t_evaporation_yr']:.2e} yr")
    print(f"{'':26s}  K_core={row['kretschmann_core_m^-4']:.2e}  K/K_Pl={row['K_core_over_K_Planck']:.2e}  "
          f"rho_core c^2={row['core_energy_density_Jm3']:.2e} J/m^3  QGP={row['reaches_QGP']}  "
          f"echo={row['echo_delay_s']:.2e} s")

kugelblitz / primordial     r_s=1.49e-15 m  tau=4.95e-24 s  T_H=1.23e+11 K  t_evap=2.67e+12 yr
                            K_core=4.93e+60  K/K_Pl=3.37e-79  rho_core c^2=6.55e+72 J/m^3  QGP=True  echo=4.55e-22 s
stellar (10 M_sun)          r_s=2.95e+04 m  tau=9.85e-05 s  T_H=6.17e-09 K  t_evap=2.10e+70 yr
                            K_core=3.15e-17  K/K_Pl=2.15e-156  rho_core c^2=1.66e+34 J/m^3  QGP=False  echo=1.78e-02 s
intermediate (1e4 M_sun)    r_s=2.95e+07 m  tau=9.85e-02 s  T_H=6.17e-12 K  t_evap=2.10e+79 yr
                            K_core=3.15e-29  K/K_Pl=2.15e-168  rho_core c^2=1.66e+28 J/m^3  QGP=False  echo=1.92e+01 s
supermassive (1e9 M_sun)    r_s=2.95e+12 m  tau=9.85e+03 s  T_H=6.17e-17 K  t_evap=2.10e+94 yr
                            K_core=3.15e-49  K/K_Pl=2.15e-188  rho_core c^2=1.66e+18 J/m^3  QGP=False  echo=2.14e+06 s


## P2–P5 — secondary

In [5]:
part = E.run('energy_partition', {'M_kg': 10*M.M_SUN})['result']
print('P2 partition:', {k: part[k] for k in ('space_fraction','matter_fraction','basis')})

rows = E.run('mass_class_table', {})['result']
qgp = [r['class'] for r in rows if r['reaches_QGP']]
print('P3 reaches QGP:', qgp, ' (+ stellar M <~ 3 Msun)')

bud = E.run('cosmic_cavitation_budget', {})['result']
print('P4 budget:', bud['omega_cavitation'], 'vs Omega_Lambda', bud['omega_lambda_obs'],
      '->', bud['verdict'])

ts = E.run('interior_timescales', {'M_kg': 10*M.M_SUN})['result']
print('P5 T_dS / T_H =', ts['T_desitter_over_T_hawking'])

P2 partition: {'space_fraction': 0.754, 'matter_fraction': 0.246, 'basis': '1 - d* (default)'}
P3 reaches QGP: ['kugelblitz / primordial']  (+ stellar M <~ 3 Msun)
P4 budget: 7.540000000000001e-06 vs Omega_Lambda 0.6847 -> falls short — dark-flow (directional) signature, not a dark-energy magnitude
P5 T_dS / T_H = 1.9999999999999998


## Scorecard

| # | prediction | verdict |
|---|---|---|
| **P1** | `K_core(M) = (3/2) c⁸/(G⁴M⁴)` — finite, `M⁻⁴`, sub-Planckian | **STANDS** (consistency check passes; sole falsifier not triggered) |
| P2 | energy split `1 - d* : d*` | ASSERTED — no independent measurement yet |
| P3 | QGP only for `M ≲ 3 M☉` + kugelblitz | CONFIRMED in-model (`ρ_core ∝ M⁻²`) |
| P4 | cosmic budget ≈ `Ω_Λ` | **FALSIFIED as a magnitude** (short ~5 orders) — kept: dark-flow signature only |
| P5 | `T_dS = 2 T_H` exactly | CONFIRMED (ratio = 2.000…) |

**The claim survives because it rests on P1 alone.** No falsified secondary was reinterpreted to save it.

### The largest gap
There is **no derivation** that nature selects the gravastar/Abrikosov interior. P1 is verified as an internal identity across a 27-order mass sweep; that is evidence of consistency, not a theorem. The observational falsifier (ringdown echoes) is live for LIGO/Virgo now and decisive for LISA on supermassive mergers.